# 01.1 — Validate V1 + V1.5 features on map

Visual inspection of all features to sanity-check the data.

> **Legacy 1 km / V1.5 workflow**
>
> This notebook is retained as a historical reproducibility reference. Use the 500 m operational pipeline (`01_collect_datasets`, `01.4`, and `06`–`10`) for current work. Do not use this notebook to overwrite current processed data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from estonia_landuse.data.constants import DATA_PROCESSED, CRS_ESTONIAN

In [ ]:
# Load features + geometry
grid = gpd.read_file(DATA_PROCESSED / "base_grid.gpkg")
features = pd.read_parquet(DATA_PROCESSED / "features_v1.parquet")

# Merge geometry with features
gdf = grid[["cell_id", "geometry"]].merge(features, on="cell_id")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=CRS_ESTONIAN)

print(f"{len(gdf)} cells, {len(gdf.columns)} columns")
gdf.head()

## Land cover proportions

In [ ]:
lc_cols = ["forest_pct", "wetland_pct", "agriculture_pct", "grassland_pct", "urban_pct", "water_pct"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, lc_cols):
    gdf.plot(column=col, ax=ax, legend=True, cmap="YlGn",
             legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
    ax.set_title(col.replace("_pct", " %"))
    ax.set_axis_off()

plt.suptitle("CORINE Land Cover Proportions per Cell", fontsize=14)
plt.tight_layout()
plt.show()

## Population and infrastructure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

gdf.plot(column="TOTAL_24", ax=axes[0], legend=True, cmap="Reds",
         legend_kwds={"shrink": 0.6})
axes[0].set_title("Population (2024)")
axes[0].set_axis_off()

gdf.plot(column="road_density_km", ax=axes[1], legend=True, cmap="Oranges",
         legend_kwds={"shrink": 0.6})
axes[1].set_title("Road Density (km/cell)")
axes[1].set_axis_off()

gdf.plot(column="building_count", ax=axes[2], legend=True, cmap="Purples",
         legend_kwds={"shrink": 0.6})
axes[2].set_title("Building Count")
axes[2].set_axis_off()

plt.suptitle("Population & Infrastructure", fontsize=14)
plt.tight_layout()
plt.show()

## Proxy scores and protected areas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

gdf.plot(column="naturalness_score", ax=axes[0, 0], legend=True, cmap="Greens",
         legend_kwds={"shrink": 0.6})
axes[0, 0].set_title("Naturalness Score")
axes[0, 0].set_axis_off()

gdf.plot(column="carbon_score", ax=axes[0, 1], legend=True, cmap="YlOrBr",
         legend_kwds={"shrink": 0.6})
axes[0, 1].set_title("Carbon Score (V1 lookup)")
axes[0, 1].set_axis_off()

gdf.plot(column="protected_overlap_pct", ax=axes[1, 0], legend=True, cmap="Blues",
         legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
axes[1, 0].set_title("Protected Area Overlap (%)")
axes[1, 0].set_axis_off()

gdf.plot(column="distance_to_protected_m", ax=axes[1, 1], legend=True, cmap="RdYlGn_r",
         legend_kwds={"shrink": 0.6})
axes[1, 1].set_title("Distance to Protected Area (m)")
axes[1, 1].set_axis_off()

plt.suptitle("Proxy Scores & Protected Areas", fontsize=14)
plt.tight_layout()
plt.show()

## Dominant land cover group

In [ ]:
group_colors = {
    "forest": "#228B22",
    "wetland": "#4682B4",
    "agriculture": "#DAA520",
    "grassland": "#90EE90",
    "urban": "#808080",
    "water": "#1E90FF",
    "other_natural": "#DEB887",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for group, color in group_colors.items():
    subset = gdf[gdf["land_cover_group"] == group]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, label=f"{group} ({len(subset)})")

ax.legend(loc="lower right")
ax.set_title("Dominant Land Cover Group")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## V1 Summary statistics

In [ ]:
numeric_cols = [
    "TOTAL_24", "forest_pct", "wetland_pct", "agriculture_pct",
    "grassland_pct", "urban_pct", "water_pct",
    "naturalness_score", "carbon_score",
    "protected_overlap_pct", "distance_to_protected_m",
    "road_density_km", "building_count",
]
gdf[numeric_cols].describe().round(3)

---
# V1.5 Carbon Features

Soil/peat, hydrology, and combined carbon scores from notebook 04.  
If you haven't run it yet, these sections will be skipped.

In [ ]:
CARBON_DIR = Path("../data/processed/carbon_v1_5")
HAS_V15 = CARBON_DIR.exists() and (CARBON_DIR / "carbon_scores.parquet").exists()

if HAS_V15:
    soil_df = pd.read_parquet(CARBON_DIR / "soil_peat_features.parquet")
    hydro_df = pd.read_parquet(CARBON_DIR / "hydrology_features.parquet")
    scores_df = pd.read_parquet(CARBON_DIR / "carbon_scores.parquet")
    print(f"Loaded: soil ({len(soil_df.columns)} cols), hydro ({len(hydro_df.columns)} cols), scores ({len(scores_df.columns)} cols)")
else:
    print("Carbon v1.5 data not found. Run notebook 04_carbon_dataset.ipynb first.")

## Soil / Peat

In [ ]:
if HAS_V15:
    gdf_soil = grid[["cell_id", "geometry"]].merge(soil_df, on="cell_id")
    gdf_soil = gpd.GeoDataFrame(gdf_soil, geometry="geometry", crs=CRS_ESTONIAN)

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    gdf_soil.plot(column="wetland_mire_overlap_pct", ax=axes[0, 0], legend=True, cmap="GnBu",
                  legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
    axes[0, 0].set_title("Wetland/Mire Overlap (%)")
    axes[0, 0].set_axis_off()

    gdf_soil.plot(column="peat_overlap_pct", ax=axes[0, 1], legend=True, cmap="YlOrBr",
                  legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
    axes[0, 1].set_title("Peat Deposit Overlap (%)")
    axes[0, 1].set_axis_off()

    gdf_soil.plot(column="soil_carbon_relevance_score", ax=axes[1, 0], legend=True, cmap="YlOrRd",
                  legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
    axes[1, 0].set_title("Soil Carbon Relevance Score")
    axes[1, 0].set_axis_off()

    gdf_soil.plot(column="wetland_restoration_soil_score", ax=axes[1, 1], legend=True, cmap="BuGn",
                  legend_kwds={"shrink": 0.6}, vmin=0, vmax=1)
    axes[1, 1].set_title("Wetland Restoration Soil Score")
    axes[1, 1].set_axis_off()

    plt.suptitle("Soil / Peat Features (V1.5)", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
if HAS_V15:
    status_colors = {
        "none": "#f0f0f0",
        "wetland_no_peat_data": "#66c2a5",
        "natural": "#1b9e77",
        "exploitable": "#d95f02",
        "damaged": "#e7298a",
    }

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    for status, color in status_colors.items():
        subset = gdf_soil[gdf_soil["peatland_status"] == status]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, label=f"{status} ({len(subset)})")

    ax.legend(loc="lower right")
    ax.set_title("Peatland Status")
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

## Hydrology

In [ ]:
if HAS_V15:
    gdf_hydro = grid[["cell_id", "geometry"]].merge(hydro_df, on="cell_id")
    gdf_hydro = gpd.GeoDataFrame(gdf_hydro, geometry="geometry", crs=CRS_ESTONIAN)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    gdf_hydro.plot(column="ditch_density_1km", ax=axes[0, 0], legend=True, cmap="Blues")
    axes[0, 0].set_title("Ditch Density (km/cell)")
    axes[0, 0].set_axis_off()

    gdf_hydro.plot(column="stream_density_1km", ax=axes[0, 1], legend=True, cmap="Blues")
    axes[0, 1].set_title("Stream Density (km/cell)")
    axes[0, 1].set_axis_off()

    gdf_hydro.plot(column="distance_to_water_m", ax=axes[0, 2], legend=True, cmap="RdYlBu")
    axes[0, 2].set_title("Distance to Water (m)")
    axes[0, 2].set_axis_off()

    gdf_hydro.plot(column="waterbody_overlap_pct", ax=axes[1, 0], legend=True, cmap="Blues",
                   vmin=0, vmax=0.5)
    axes[1, 0].set_title("Waterbody Overlap (%)")
    axes[1, 0].set_axis_off()

    gdf_hydro.plot(column="water_proximity_score", ax=axes[1, 1], legend=True, cmap="GnBu",
                   vmin=0, vmax=1)
    axes[1, 1].set_title("Water Proximity Score")
    axes[1, 1].set_axis_off()

    gdf_hydro.plot(column="hydrology_restoration_score", ax=axes[1, 2], legend=True, cmap="YlGnBu",
                   vmin=0, vmax=1)
    axes[1, 2].set_title("Hydrology Restoration Score")
    axes[1, 2].set_axis_off()

    plt.suptitle("Hydrology Features (V1.5)", fontsize=14)
    plt.tight_layout()
    plt.show()

## Combined Carbon Scores

In [ ]:
if HAS_V15:
    gdf_scores = grid[["cell_id", "geometry"]].merge(scores_df, on="cell_id")
    gdf_scores = gpd.GeoDataFrame(gdf_scores, geometry="geometry", crs=CRS_ESTONIAN)

    score_cols = [
        "carbon_stock_score",
        "protect_carbon_benefit",
        "afforestation_carbon_potential",
        "wetland_restoration_carbon_potential",
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, col in zip(axes.flat, score_cols):
        vmax = gdf_scores[col].quantile(0.95)
        gdf_scores.plot(column=col, ax=ax, legend=True, cmap="YlOrRd",
                        legend_kwds={"shrink": 0.6}, vmin=0, vmax=max(vmax, 0.01))
        ax.set_title(col.replace("_", " ").title())
        ax.set_axis_off()

    plt.suptitle("Combined Carbon Scores (V1.5)", fontsize=14)
    plt.tight_layout()
    plt.show()

## V1.5 Summary statistics

In [ ]:
if HAS_V15:
    print("=== Soil/Peat ===")
    display(soil_df.select_dtypes(include='number').describe().round(3))
    print("\n=== Hydrology ===")
    display(hydro_df.describe().round(3))
    print("\n=== Carbon Scores ===")
    display(scores_df.select_dtypes(include='number').describe().round(3))